In [ ]:
### S01 · Create, filter and select ★


#Create a DataFrame and return just `name` and `age` for London residents aged 25 or more. 
# Expected: `(Cara,29)`.


from pyspark.sql.functions import col

rows = [('Alice',24,'London'),('Bob',32,'Paris'),('Cara',29,'London')]

columns = [ "name", "age", "city" ] 


df = spark.createDataFrame( rows, schema = columns )

df2 = (
    df.filter(
        (col("city") == "London") &
        (col("age") >= 25)
    )
    .select("name", "age")
)

df2.show()
df2.display()


In [ ]:
### S02 · Add a conditional column ★

rows = [('A',40000),('B',65000),('C',90000)]

columns =  ('employee','salary')

#Add `band`: `"high"` for salary >= 70000, `"medium"` for >= 50000, otherwise `"low"`. 
# Expected bands: low, medium, high.

from pyspark.sql.functions import col, when

df = spark.createDataFrame( rows, columns)

df2 = ( df.withColumn("band", 
                    when(col("salary") >= 70000, "high")
                    .when(col("salary") >= 50000, "medium")
                    .otherwise("low"))
)
df2.show()
df2.display()

In [ ]:
### S03 · Clean names and null values ★★

rows = [(' alice ',None),('BOB',45),(' Cara ',30)]

columns = ("name","age") 

#Create `clean_name` with trimmed, title-cased names; 
# replace null ages with `0`. Expected clean names: Alice, Bob, Cara.

from pyspark.sql.functions import col, when, isnull, trim, initcap

df = spark.createDataFrame( rows, columns)

#clean_name = df.withColumn("age", when( isnull(col("age")) , 0).otherwise( col("age")) )

clean_name = ( df.fillna(  { "age":  0} )
            .withColumn("clean_name", trim(col("name")))
            .withColumn("clean_name", initcap( col("clean_name" )   ))
            )
clean_name.display()

clean_name.show()

In [ ]:

## S04 - count events by type


rows = [
    ("login",),
    ("click",),
    ("login",),
    ("purchase",),
    ("click",),
    ("login",)
]
import pyspark.sql.functions as F

columns = ["event"]

#Return the number of occurrences of each event, 
# sorted by count descending and then event name ascending.

df = spark.createDataFrame( rows, columns)

df_agg = df.groupBy("event").agg( F.count('*').alias("occurences") )

df_agg = df_agg.orderBy( F.col("occurences").desc(), F.col("event").asc() )
df_agg.display()
df_agg.show()



In [ ]:
### S05 · Group with multiple aggregates ★★


rows = [    ("IT", 60000),    ("IT", 80000),    ("HR", 45000),    ("HR", 55000)]
columns = ["dept", "salary"]


# Create a DataFrame and return the following information for each department:

# - `dept` — department name
# - `employee_count` — number of employees
# - `average_salary` — average salary
# - `max_salary` — highest salary

# Sort by department name ascending.

# Expected output:

# ```
# +----+--------------+--------------+----------+
# |dept|employee_count|average_salary|max_salary|
# +----+--------------+--------------+----------+
# |  HR|             2|       50000.0|     55000|
# |  IT|             2|       70000.0|     80000|
# +----+--------------+--------------+----------+

#Hint: You can call `agg()` once with multiple aggregation expressions.

#Try solving it from memory using your `F` import convention.

# This is a particularly useful pattern for data engineering interviews because you're often asked to produce summary statistics from large datasets.


import pyspark.sql.functions as F

df = spark.createDataFrame( rows, columns)

df_agg = (df.groupBy("dept")
          .agg( F.count('*').alias("employee_count"), 
               F.avg("salary").alias("average_salara"), 
               F.max("salary").alias("max_salary") ) 
          .orderBy("dept")
)


df_agg.collect()
df_agg.show()


In [ ]:
# S06 · Deduplicate using a business key ★★

rows = [
    (1, "a@example.com"),
    (2, "b@example.com"),
    (3, "a@example.com")
]

columns = ["id", "email"]

# Create a DataFrame and return unique email addresses only, sorted alphabetically.

# Expected output:
# +-------------+
# |        email|
# +-------------+
# |a@example.com|
# |b@example.com|
# +-------------+


# Hint: Consider distinct() or dropDuplicates().



df = spark.createDataFrame( rows, columns)

df2 = df.select("email").distinct().orderBy("email")

#df2 = df.dropDuplicates( ["email"])
df2.show()


# Bonus interview question: If you use dropDuplicates(["email"]) while retaining the id column, which ID would Spark keep for a@example.com — 1 or 3?

# Try answering that as well as writing the code.

# answer : there is no guarantee

# Interview takeaway: distinct() and dropDuplicates() are ideal for removing duplicates, but don't rely on dropDuplicates() to choose a particular surviving record.

In [ ]:
#07 · Inner and left joins ★★

customers = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Cara")
]

orders = [
    (101, 1, 20),
    (102, 1, 30),
    (103, 2, 15)
]

df_customers = spark.createDataFrame(
    customers, ["customer_id", "name"]
)

df_orders = spark.createDataFrame(
    orders, ["order_id", "customer_id", "amount"]
)

# Task: Write PySpark code using a left join to return each customer's name and their order amounts, including customers who have no orders.

# Expected output (order of rows may vary):
# +-----+------+
# | name|amount|
# +-----+------+
# |Alice|    20|
# |Alice|    30|
# |  Bob|    15|
# | Cara|  NULL|
# +-----+------+

import pyspark.sql.functions as F

# the left table is the one before the .join
df = df_customers.join(df_orders, on = "customer_id", how="left").select("name", "amount")

df.show()

# Alice appears twice as there are two entries for Alice in the orders table

In [ ]:
# S08 · Customers with no orders ★★
# Use the same two DataFrames from S07:
# df_customersdf_orders


# Task: Return only customers who have never placed an order.

# Expected result:
# +-----------+----+
# |customer_id|name|
# +-----------+----+
# |          3|Cara|
# +-----------+----+


# Hint: PySpark has a join type called left_anti, which is specifically designed for this.

# Bonus interview question: What's the difference between a left join and a left_anti join?

# Try writing the solution and explaining that distinction.

df = df_customers.join(df_orders, on = "customer_id", how="left_anti")

# essentially a left anti join returns the rows that had no match from the standard left join, these are the rows
# that would have resulted in nulls.

df.show()


In [ ]:
# S09 · Revenue by customer (including zero) ★★


# Using the same DataFrames:
# df_customers,df_orders



# Task: Return each customer's total order amount, including customers without orders, whose total should be 0.

# Expected output:
# +-----+------------+
# | name|total_amount|
# +-----+------------+
# |Alice|          50|
# |  Bob|          15|
# | Cara|           0|
# +-----+------------+


# Sort by name ascending.

# Hints: You'll need join(), groupBy(), sum() and a way of handling nulls.

# Bonus interview question: Why does Cara's total initially become NULL rather than 0 when you use sum("amount") after a left join?

# Try solving it from memory, using  import pyspark.sql.functions as F


import pyspark.sql.functions as F

df = df_customers.join( df_orders, on = "customer_id", how="left").fillna({"amount":0}  )
# we use a left join on df_customers so we have all customers, so Cara's amount column will be null


df2 = df.groupBy("name").agg( F.sum( F.col("amount")  ).alias("total_amount")  ).orderBy( F.col("name").asc() )

df2.show()

In [ ]:
# S10 · Latest event per device ★★★
# We're moving on to window functions, one of the most important intermediate-to-advanced PySpark interview topics.

# Given:

rows = [    ("A", "2026-01-01 09:00", "ok"),    ("A", "2026-01-02 08:00", "fail"),    ("B", "2026-01-01 10:00", "ok")]
columns = ["device", "timestamp", "status"]



# Task: Return only the latest event for each device.

# Requirements:
# 1. Convert the string timestamp column to a proper Spark timestamp type.
# 2. Use row_number() with a window partitioned by device.
# 3. Order timestamps descending within each partition.
# 4. Keep only the latest record for each device.
# Expected result:
# +------+-------------------+------+
# |device|timestamp          |status|
# +------+-------------------+------+
# |A     |2026-01-02 08:00:00|fail  |
# |B     |2026-01-01 10:00:00|ok    |
# +------+-------------------+------+


# Hints: Window.partitionBy(), orderBy(), row_number() and to_timestamp().

# Bonus interview question: What's the difference between row_number(), rank() and dense_rank() when two records have the same timestamp?

# Give it a try from memory, and we'll work through any unfamiliar window syntax.


import pyspark.sql.functions as F
from pyspark.sql import Window

initial = spark.createDataFrame( rows, columns)

latest_event = (initial
                .withColumn("timestamp", F.to_timestamp("timestamp"))
                .withColumn("rank", F.row_number().over( Window.partitionBy("device").orderBy( F.col("timestamp").desc()     )))
                .filter( F.col("rank") == 1)
                .drop("rank")


)

latest_event.show()

# the difference between row_number(), rank() and dense_rank() is how they deal with duplicates.  row_number() gives 
# each row a unique id, rank() gives duplicates the same id, but preserves the overall numbering, and dense_rank()
# also labels duplicates with the same id, but it does not preserve the overall numbering

In [ ]:
### S11 · Rank salaries within departments ★★★

# Given:


rows = [    ("IT", "Alice", 70000),    ("IT", "Bob", 80000),    ("IT", "Cara", 80000),    ("HR", "Dan", 50000) ]

columns = ["dept", "name", "salary"]


# Create a DataFrame and rank employees by salary within each department, highest salary first.

# This time, use `dense_rank()` so employees with equal salaries receive the same rank.

# Expected output:

# 
# +----+-----+------+----+
# |dept| name|salary|rank|
# +----+-----+------+----+
# |  HR|  Dan| 50000|   1|
# |  IT|  Bob| 80000|   1|
# |  IT| Cara| 80000|   1|
# |  IT|Alice| 70000|   2|
# +----+-----+------+----+
# 

# Sort by `dept` ascending, `rank` ascending, and `name` ascending.

# Bonus interview question: Why would using `row_number()` instead of `dense_rank()` produce an incorrect result for Bob and Cara?

# You already have nearly all the syntax from S10, so try this one without looking anything up.

import pyspark.sql.functions as f
from pyspark.sql import Window


df = spark.createDataFrame( rows, columns)

df2 = df.withColumn( "rank", 
                    f.dense_rank().over( Window.partitionBy(f.col("dept") ).orderBy( f.col("salary").desc() ) ) 
        ).orderBy( f.col("dept").asc(),f.col("rank").asc(),f.col("name").asc()  )
                                        

df2.show()                              

# row_number() will allocate them unique ranks, when in fact they are the same.


In [ ]:
### S12 · Null-safe filtering ★★

import pyspark.sql.functions as f

Rows = [('A',60),('B',None), ('C',85),('D',50) ] 


#
# Return count of records with score >= 60 and separately count records with null score. 
# Expected 2 and 1. 
# 
# Explain why a normal comparison does not count null rows.

df = spark.createDataFrame( Rows, ["name","score" ])

count = df.filter( f.col("score") >= 60 ).count()

print( f"60 and over : {count} " )



count = df.filter( f.col("score").isNull() ).count()


print( f"nulls : {count} " )

# null isn't a valid valid so it will never be included in any normal filter.

In [ ]:
# S13 · Parse a JSON string column ★★★
# This is particularly relevant for data engineering and AI engineering pipelines, where API responses and event messages frequently contain JSON.

# Given:


rows = [    (1, '{"type":"click","value":3}'),    (2, '{"type":"view","value":1}')]
columns = ["id", "payload"]


# Create a DataFrame and:
# 1. Parse the JSON string in payload into a structured column using from_json().
# 2. Define an explicit schema using StructType, StructField, StringType and IntegerType.
# 3. Extract the fields type and value.
# 4. Return only id, type and value.
# Expected output:
# +---+-----+-----+
# | id| type|value|
# +---+-----+-----+
# |  1|click|    3|
# |  2| view|    1|
# +---+-----+-----+


# Hints: You'll need these imports:
# import pyspark.sql.functions as f
# from pyspark.sql.types import (    StructType, StructField, StringType, IntegerType)


# The unfamiliar part may be defining the schema. Here's the basic pattern:
# schema = StructType([    StructField("field_name", StringType(), True)])

# For extracting a nested field, remember that PySpark supports dot notation:
# f.col("parsed.type")

import pyspark.sql.functions as f
from pyspark.sql.types import (    StructType, StructField, StringType, IntegerType)

json_schema = StructType( [ StructField("type", StringType(), True), StructField("value", IntegerType(), True)] )

df = spark.createDataFrame( rows, columns )

df_json = df.withColumn( "parsed", f.from_json( f.col("payload"), json_schema ) )

df_json.printSchema()

#df_json.display()

df_final = df_json.select( f.col("id"), f.col("parsed.type"), f.col("parsed.value"))
df_final.show()

# root
#  |-- id: long (nullable = true)
#  |-- payload: string (nullable = true)
#  |-- parsed: struct (nullable = true)
#  |    |-- type: string (nullable = true)
#  |    |-- value: integer (nullable = true)

# +---+-----+-----+
# | id| type|value|
# +---+-----+-----+
# |  1|click|    3|
# |  2| view|    1|
# +---+-----+-----+



# Bonus interview question: Why is it generally preferable to parse JSON into a structured column rather than repeatedly extracting individual fields from the original JSON string?

# just for performance reasons I imagine
